# 🤖 머신러닝 실습: 영화 리뷰 감성 분석

## 🎯 학습 목표
- 머신러닝의 기본 개념을 이해합니다
- 텍스트 데이터 전처리 과정을 학습합니다
- 감성 분석(Sentiment Analysis)을 실습합니다
- 실제 데이터로 예측 모델을 만들어봅니다

## 💡 감성 분석이란?
**감성 분석(Sentiment Analysis)**은 텍스트에서 감정이나 의견을 자동으로 분류하는 기술입니다.

### 🌟 실생활 활용 예시
- **소셜미디어**: 트위터, 인스타그램 댓글 분석
- **쇼핑몰**: 상품 리뷰 분석으로 평점 예측
- **뉴스**: 기사에 대한 여론 분석
- **고객 서비스**: 문의 내용의 긴급도 판단

### 📊 오늘 우리가 할 일
1. **영화 리뷰 데이터** 불러오기
2. **한국어 텍스트 전처리** (형태소 분석)
3. **워드클라우드** 만들기
4. **머신러닝 모델** 훈련하기
5. **새로운 리뷰** 예측해보기

---

### ➡️ 파이프라인    

- 데이터 로딩  (영화 리뷰 수집) -> 전처리  (결측·중복 제거 & 길이 분석) -> 토큰화 (형태소 분석) -> 벡터화(TF-IDF 변환) -> 모델 학습  (Logistic Regression) -> 예측 & 평가(정확도, F1, 혼동 행렬)

---

## 🔧 1단계: 환경 설정

**📦 필요한 라이브러리 설치:**
- **konlpy**: 한국어 자연어 처리
- **wordcloud**: 단어 구름 시각화
- **scikit-learn**: 머신러닝 모델
- **xgboost**: 고성능 머신러닝 알고리즘
- **shap**: 모델 해석 도구

**주의**: 이 셀은 최초 1회만 실행하면 됩니다!

In [ ]:
# 🔧 환경 설정 - 최초 1회만 실행하세요 (라이브러리 설명은 위 셀 참고)
!pip install konlpy wordcloud scikit-learn xgboost shap tqdm unidic-lite
!pip install koreanize-matplotlib

!apt-get update -y
!apt-get install -y fonts-nanum
!fc-cache -fv

print("✅ 라이브러리 설치가 완료되었습니다!")

## 📊 2단계: 데이터 불러오기

**🎬 NSMC 데이터셋이란?**
- **N**aver **S**entiment **M**ovie **C**orpus
- 네이버 영화 리뷰 데이터
- 약 15만 개의 영화 리뷰
- 각 리뷰는 긍정(1) 또는 부정(0)으로 라벨링됨

**📁 데이터 구조:**
- `id`: 리뷰 고유 번호
- `document`: 실제 리뷰 텍스트
- `label`: 0(부정) 또는 1(긍정)


In [ ]:
# 📚 라이브러리 불러오기
# 데이터 처리
import numpy as np       # 수치 계산
import pandas as pd      # 표 형태 데이터(데이터프레임)

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# 한국어 형태소 분석
from konlpy.tag import Okt

# 진행 표시줄
from tqdm.notebook import tqdm
tqdm.pandas(desc="토큰화 진행")

# 한글 폰트
import koreanize_matplotlib
%matplotlib inline

import matplotlib.font_manager as fm
import matplotlib as mpl

# 경고 메시지 숨기기
import warnings
warnings.simplefilter('ignore', category=UserWarning)
warnings.simplefilter('ignore', category=FutureWarning)

print("✅ 모든 라이브러리가 성공적으로 불러와졌습니다!")
print(f"📦 Pandas 버전: {pd.__version__}")
print(f"📦 NumPy 버전: {np.__version__}")

In [ ]:
# ============================================================
# 📥 데이터 불러오기
# ============================================================
# NSMC(Naver Sentiment Movie Corpus) 데이터셋 사용
# - 네이버 영화 리뷰 약 15만 개
# - label 0: 부정적 리뷰, label 1: 긍정적 리뷰

# GitHub에서 데이터 다운로드
url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"

# pd.read_csv(): CSV/텍스트 파일을 데이터프레임으로 읽기
# sep='\t': 탭으로 구분된 파일
# dropna(): 빈 값(결측치) 제거
df = pd.read_csv(url, sep='\t').dropna()

# 데이터 기본 정보 확인
print("=" * 50)
print("📊 데이터 기본 정보")
print("=" * 50)
print(f"📌 전체 리뷰 수: {len(df):,}개")
print(f"📌 컬럼(열) 목록: {list(df.columns)}")
print(f"📌 긍정 리뷰 수: {(df['label'] == 1).sum():,}개")
print(f"📌 부정 리뷰 수: {(df['label'] == 0).sum():,}개")
print("=" * 50)

# 처음 5개 데이터 미리보기
print("\n🔍 데이터 미리보기 (처음 5개):")
df.head()

## 🔤 3단계: 텍스트 전처리 (토큰화)

**🧠 형태소 분석이란?**
- 문장을 의미있는 단위로 나누는 과정
- 예: "영화가 재밌어요" → ["영화", "재밌다"]

**⚙️ 전처리 과정:**
1. **형태소 분석**: 단어를 기본 형태로 변환
2. **불용어 제거**: '의', '가', '이' 같은 의미없는 단어 제거
3. **짧은 단어 제거**: 1글자 단어 제거

**📈 샘플링하는 이유:**
- 전체 15만 개 데이터는 처리 시간이 오래 걸림
- 10% 샘플링으로 빠른 실습 진행


In [ ]:
# 🔍 데이터 탐색 및 전처리
# 1) 리뷰 길이(글자 수) 계산
df['length'] = df['document'].str.len()

# 2) 기본 통계
print("=" * 50)
print("📊 데이터 기본 통계")
print("=" * 50)
print(df.describe())

# 3) 중복 리뷰 확인 후 제거
dup_ratio = df.duplicated('document').mean()
print(f"\n⚠️ 중복 리뷰 비율: {dup_ratio:.2%}")
print(f"   중복 리뷰 수: {df.duplicated('document').sum():,}개")

before_count = len(df)
df = df.drop_duplicates('document')
after_count = len(df)

print(f"\n✅ 중복 제거 완료!")
print(f"   제거 전: {before_count:,}개 → 제거 후: {after_count:,}개")
print(f"   제거된 리뷰: {before_count - after_count:,}개")

In [ ]:
# ============================================================
# 📊 긍정/부정 리뷰 비교 분석
# ============================================================
# 💡 긍정/부정 리뷰의 특성을 파악하면 모델 성능을 예측할 수 있어요!

# groupby(): 특정 컬럼을 기준으로 데이터 그룹화
# agg(): 여러 통계 함수를 한번에 적용
print("=" * 60)
print("📊 긍정/부정별 문장 길이 통계")
print("=" * 60)

length_stats = df.groupby('label')['length'].agg([
    'count',   # 개수
    'mean',    # 평균
    'median',  # 중앙값
    'std',     # 표준편차 (데이터가 얼마나 퍼져있는지)
    'min',     # 최솟값
    'max'      # 최댓값
])
length_stats.index = ['부정 리뷰 (0)', '긍정 리뷰 (1)']
print(length_stats.round(2))

# 💡 해석 도움말
print("\n💡 용어 설명:")
print("   - count: 리뷰 개수")
print("   - mean: 평균 글자 수")
print("   - median: 중앙값 (데이터를 정렬했을 때 가운데 값)")
print("   - std: 표준편차 (값이 클수록 길이가 다양함)")


In [ ]:
# ============================================================
# 📈 시각화 1: 문장 길이 분포 비교
# ============================================================
# 💡 히스토그램: 데이터의 분포를 막대 그래프로 표현

# 그래프 크기 설정
plt.figure(figsize=(12, 5))
plt.rcParams['font.size'] = 12

# 히스토그램 그리기
# bins=50: 막대를 50개로 나눔
# alpha=0.7: 투명도 (0~1, 1이 완전 불투명)
# density=True: 밀도로 표현 (합이 1이 되도록)
df[df['label']==0]['length'].hist(bins=50, alpha=0.7, label='부정 (0)', color='lightcoral', density=True)
df[df['label']==1]['length'].hist(bins=50, alpha=0.7, label='긍정 (1)', color='lightblue', density=True)

plt.xlabel('문장 길이 (글자 수)')
plt.ylabel('밀도 (비율)')
plt.title('📊 긍정/부정별 문장 길이 분포')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 해석: 두 분포가 비슷하면 문장 길이만으로는 긍정/부정을 구분하기 어렵습니다.")

In [ ]:
plt.figure(figsize=(20, 10))
plt.rcParams['font.size'] = 12

# 1. 길이별 샘플 수 분포
plt.subplot(2, 3, 5)
length_counts = df['length'].value_counts().sort_index()
plt.plot(length_counts.index, length_counts.values, color='green', alpha=0.7)
plt.xlabel('문장 길이 (글자 수)')
plt.ylabel('샘플 수')
plt.title('문장 길이별 샘플 수 분포')
plt.grid(True, alpha=0.3)


In [ ]:
plt.figure(figsize=(20, 10))
plt.rcParams['font.size'] = 12

# 8. 극단값 분석 (매우 짧은/긴 문장들)
plt.subplot(2, 3, 6)
# 상위 5%, 하위 5% 문장들의 감성 분포
short_threshold = df['length'].quantile(0.05)
long_threshold = df['length'].quantile(0.95)

short_pos_ratio = df[df['length'] <= short_threshold]['label'].mean()
normal_pos_ratio = df[(df['length'] > short_threshold) & (df['length'] < long_threshold)]['label'].mean()
long_pos_ratio = df[df['length'] >= long_threshold]['label'].mean()

categories = ['매우 짧은\n문장\n(하위 5%)', '일반적인\n문장\n(중간 90%)', '매우 긴\n문장\n(상위 5%)']
ratios = [short_pos_ratio, normal_pos_ratio, long_pos_ratio]
colors = ['lightcoral', 'lightgreen', 'lightblue']

bars = plt.bar(categories, ratios, color=colors, alpha=0.7)
plt.ylabel('긍정 비율')
plt.title('📊 문장 길이 극단값별 긍정 비율')
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7)
plt.grid(True, alpha=0.3)

# 각 막대 위에 수치 표시
for bar, ratio in zip(bars, ratios):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 9. 통계적 유의성 검정
from scipy import stats
neg_lengths = df[df['label']==0]['length']
pos_lengths = df[df['label']==1]['length']

# t-검정 실시
t_stat, p_value = stats.ttest_ind(neg_lengths, pos_lengths)

print("\n📈 통계적 분석 결과")
print("="*50)
print(f"부정 리뷰 평균 길이: {neg_lengths.mean():.2f} ± {neg_lengths.std():.2f}")
print(f"긍정 리뷰 평균 길이: {pos_lengths.mean():.2f} ± {pos_lengths.std():.2f}")
print(f"평균 차이: {pos_lengths.mean() - neg_lengths.mean():.2f}")
print(f"t-검정 통계량: {t_stat:.4f}")
print(f"p-값: {p_value:.2e}")
print(f"통계적 유의성: {'유의함 (p < 0.05)' if p_value < 0.05 else '유의하지 않음 (p >= 0.05)'}")


In [ ]:
# 10. 실용적 인사이트 제공
print("\n💡 주요 인사이트")
print("="*50)

if pos_lengths.mean() > neg_lengths.mean():
    print(f"✅ 긍정 리뷰가 부정 리뷰보다 평균 {pos_lengths.mean() - neg_lengths.mean():.1f}글자 더 깁니다.")
else:
    print(f"✅ 부정 리뷰가 긍정 리뷰보다 평균 {neg_lengths.mean() - pos_lengths.mean():.1f}글자 더 깁니다.")

# 분위수 비교
print(f"✅ 문장 길이 중앙값 - 부정: {neg_lengths.median():.0f}글자, 긍정: {pos_lengths.median():.0f}글자")
print(f"✅ 매우 짧은 문장(≤{short_threshold:.0f}글자) 긍정 비율: {short_pos_ratio:.1%}")
print(f"✅ 매우 긴 문장(≥{long_threshold:.0f}글자) 긍정 비율: {long_pos_ratio:.1%}")


In [ ]:

# 부정 리뷰 예시
short_neg = df[(df['length'] <= 10) & (df['label'] == 0)]['document'].head(3)
print("🔸 부정 리뷰 예시:")
for i, text in enumerate(short_neg, 1):
    print(f"   {i}. \"{text}\" ({len(text)}글자)")

# 매우 짧은 긍정 리뷰
short_pos = df[(df['length'] <= 10) & (df['label'] == 1)]['document'].head(3)
print("\n🔸 긍정 리뷰 예시:")
for i, text in enumerate(short_pos, 1):
    print(f"   {i}. \"{text}\" ({len(text)}글자)")

In [ ]:
# 🔤 텍스트 토큰화 (형태소 분석)
# 토큰화: 문장을 의미있는 단어 단위로 나누는 것 ("영화가 재밌어요" → ["영화","재밌다"])

# 1) 형태소 분석기 초기화
okt = Okt()

# 2) 불용어(분석에서 제외할 단어) 정의
stopwords = set([
    '의', '가', '이', '은', '들', '는', '좀', '잘',
    '걍', '과', '도', '를', '으로', '자', '에', '와', '한', '하다'
])

# 3) 토큰화 함수 정의
def tokenize(text):
    """문장을 토큰(단어) 리스트로 변환 (어간 추출 stem, 정규화 norm 적용)"""
    tokens = []
    for word, pos in okt.pos(text, stem=True, norm=True):
        # 불용어가 아니고 2글자 이상인 단어만 사용
        if word not in stopwords and len(word) > 1:
            tokens.append(word)
    return tokens

# 토큰화 예시
print("=" * 50)
print("🔤 토큰화 예시")
print("=" * 50)
example_text = "이 영화 정말 재밌어요! 강력 추천합니다."
print(f"원본 문장: {example_text}")
print(f"토큰화 결과: {tokenize(example_text)}")
print()

# 4) 전체 데이터에 적용 (시간 절약을 위해 10%만 샘플링)
print("⏳ 토큰화 진행 중... (약 1~2분 소요)")
sample_df = df.sample(frac=0.1, random_state=42).copy()
sample_df['tokens'] = sample_df['document'].progress_apply(tokenize)

print(f"\n✅ 토큰화 완료! (총 {len(sample_df):,}개 리뷰)")

<div style="margin:20px 0 4px;padding:14px 20px;border-radius:10px;background:#F4E7DF;border:1px solid #E7DCC6;">
<span style="font-size:19px;font-weight:800;color:#23211C;">✏️ 직접 풀어 보기 — 연습문제</span><br>
<span style="color:#5A5448;font-size:14px;line-height:1.7;">각 문제를 먼저 <b>스스로</b> 채워 본 뒤 <b>정답 노트북</b>과 비교하세요. 위에서 만든 변수(<code>tokenize</code>, <code>tfidf</code>, <code>X_tr</code>, <code>y_tr</code>, <code>clf</code>, <code>predict_sentiment</code> 등)를 그대로 씁니다. 막히면 위쪽 해당 셀을 참고하세요.</span>
</div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 1 토큰화 직접 해보기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">빈칸 채우기</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 내가 쓴 영화평 한 문장을 <code>tokenize()</code>로 토큰화하고, 불용어 목록에 단어 하나(예: <code>'영화'</code>)를 추가한 뒤 결과가 어떻게 달라지는지 확인하세요.<br>
<b>힌트</b> &nbsp; <code>tokenize('문장')</code> · 집합에 추가는 <code>stopwords.add('단어')</code><br>
<b>예상</b> &nbsp; 추가한 단어가 토큰 결과에서 사라진다
</div>
</div>

In [ ]:
# ✏️ 연습문제 1 — 빈칸(____)을 채워 보세요
my_review = "이 영화 연출이 정말 좋았어요"
print("토큰화 전:", ____(my_review))      # 힌트: 토큰화 함수

stopwords.add("____")                     # 힌트: 제외할 단어 하나 (예: 영화)
print("불용어 추가 후:", tokenize(my_review))

## ☁️ 4단계: 워드클라우드 시각화

**💭 워드클라우드란?**
- 텍스트에서 자주 나오는 단어를 크기로 표현한 시각화
- 단어가 클수록 더 자주 등장함
- 데이터의 특성을 한눈에 파악 가능

**🎯 목적:**
- 영화 리뷰에서 어떤 단어들이 많이 사용되는지 확인
- 긍정/부정 단어의 패턴 파악

In [ ]:
# ============================================================
# ☁️ 워드클라우드 만들기
# ============================================================
# 💡 워드클라우드: 자주 나오는 단어를 크게 표시하는 시각화
from collections import Counter

# 1️⃣ 모든 토큰을 하나의 리스트로 합치기
# sum(리스트, []): 중첩 리스트를 하나로 펼침
all_tokens = sum(sample_df['tokens'], [])
print(f"📊 총 토큰 수: {len(all_tokens):,}개")

# 2️⃣ 단어 빈도수 계산
# Counter: 각 항목이 몇 번 나왔는지 세어줌
word_freq = Counter(all_tokens)

# 상위 10개 단어 출력
print("\n🔝 가장 많이 나온 단어 Top 10:")
for i, (word, count) in enumerate(word_freq.most_common(10), 1):
    print(f"   {i}. {word}: {count:,}회")

# 3️⃣ 워드클라우드 생성 및 시각화
wc = WordCloud(
    font_path='/usr/share/fonts/truetype/nanum/NanumGothic.ttf',  # 한글 폰트
    background_color='white',  # 배경색
    width=800,                 # 너비
    height=400,                # 높이
    max_words=200              # 최대 단어 수
)

plt.figure(figsize=(14, 7))
plt.imshow(wc.generate_from_frequencies(word_freq))
plt.axis('off')  # 축 숨기기
plt.title('🎬 영화 리뷰 워드클라우드', fontsize=16)
plt.tight_layout()
plt.show()

print("\n💡 해석: 큰 단어일수록 리뷰에서 자주 언급된 단어입니다.")

## 🔢 5단계: 텍스트를 숫자로 변환 (벡터화)

**🤖 머신러닝의 한계:**
- 컴퓨터는 텍스트를 직접 이해할 수 없음
- 모든 데이터를 숫자로 변환해야 함

**📊 TF-IDF란?**
- **Term Frequency - Inverse Document Frequency**
- 단어의 중요도를 수치로 계산하는 방법
- 자주 나오지만 모든 문서에 공통으로 나오는 단어는 중요도를 낮춤

**⚙️ 과정:**
1. 각 단어를 숫자로 변환
2. 단어의 중요도 계산
3. 문서를 숫자 벡터로 표현

In [ ]:
# 🔢 TF-IDF 벡터화 (텍스트 → 숫자)
# 자주 나오지만 여러 문서에 공통으로 등장하면 중요도를 낮추는 가중치 방식
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1) 벡터화 객체 생성
tfidf = TfidfVectorizer(
    tokenizer=lambda x: x,  # 이미 토큰화됨
    lowercase=False,        # 한글이라 소문자 변환 불필요
    max_features=20000      # 상위 20,000개 단어만 사용
)

# 2) 벡터화 수행
X = tfidf.fit_transform(sample_df['tokens'])  # 특성
y = sample_df['label']                         # 정답(레이블)

print("=" * 50)
print("🔢 TF-IDF 벡터화 결과")
print("=" * 50)
print(f"📌 X의 형태: {X.shape}")
print(f"   → {X.shape[0]:,}개의 리뷰, {X.shape[1]:,}개의 단어 특성")
print(f"📌 y의 형태: {y.shape}")

# 3) 학습/검증 데이터 분리 (검증용 20%)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n📊 데이터 분리 결과:")
print(f"   - 학습 데이터: {X_tr.shape[0]:,}개 ({X_tr.shape[0]/len(y)*100:.0f}%)")
print(f"   - 검증 데이터: {X_val.shape[0]:,}개 ({X_val.shape[0]/len(y)*100:.0f}%)")

feature_names = tfidf.get_feature_names_out()
print(f"\n📝 단어 특성 예시 (처음 10개): {list(feature_names[:10])}")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 2 특정 단어의 TF-IDF 특성 찾기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">빈칸 채우기</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; TF-IDF가 만든 단어 특성 목록(<code>feature_names</code>)에서 단어 <code>'재밌다'</code>가 <b>몇 번째 특성</b>인지, 전체 특성이 <b>몇 개</b>인지 출력하세요.<br>
<b>힌트</b> &nbsp; 목록에서 위치 찾기 <code>list(feature_names).index('단어')</code> · 개수 <code>len(...)</code><br>
<b>예상</b> &nbsp; 인덱스 번호 1개 · 전체 특성 수(최대 20,000)
</div>
</div>

In [ ]:
# ✏️ 연습문제 2 — 빈칸을 채워 보세요
words = list(feature_names)
print("'재밌다' 특성 위치:", words.____("재밌다"))   # 힌트: 목록에서 위치 찾기
print("전체 단어 특성 수:", ____(feature_names))      # 힌트: 개수 세는 함수

## 🧠 6단계: 머신러닝 모델 훈련

**🎯 로지스틱 회귀(Logistic Regression)란?**
- 분류 문제를 해결하는 머신러닝 알고리즘
- 0과 1 사이의 확률로 결과를 예측
- 해석하기 쉽고 빠른 학습이 장점

**📊 성능 지표:**
- **Accuracy**: 전체 중 맞춘 비율 (80.2% 달성!)
- **F1 Score**: 정밀도와 재현율의 조화평균
- **Confusion Matrix**: 예측 결과의 상세 분석

**🎉 결과 해석:**
- 약 80%의 정확도로 리뷰의 감성을 예측


In [ ]:
# 🎯 베이스라인 모델: 로지스틱 회귀
# 베이스라인 = 성능 개선의 기준이 되는 가장 기본 모델
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

print("=" * 50)
print("🎯 베이스라인 모델: 로지스틱 회귀 (Logistic Regression)")
print("=" * 50)

# 1) 모델 생성 및 학습
clf = LogisticRegression(max_iter=3000)
print("⏳ 모델 학습 중...")
clf.fit(X_tr, y_tr)
print("✅ 모델 학습 완료!")

# 2) 예측
pred = clf.predict(X_val)

# 3) 성능 평가
accuracy = accuracy_score(y_val, pred)
f1 = f1_score(y_val, pred)
print(f"\n📊 성능 평가 결과:")
print(f"   - Accuracy (정확도): {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   - F1 Score: {f1:.4f}")

# 4) 혼동 행렬 시각화 (대각선=정답, 비대각선=오답)
plt.figure(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_val, pred,
    display_labels=['부정 (0)', '긍정 (1)'],
    cmap='Blues'
)
plt.title('🎯 베이스라인 모델 - 혼동 행렬', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 🔍 모델이 학습한 중요 단어 분석
# ============================================================
# 💡 로지스틱 회귀의 장점: 어떤 단어가 예측에 영향을 주는지 알 수 있음
#    - 양수 계수: 긍정에 기여하는 단어
#    - 음수 계수: 부정에 기여하는 단어

# 1️⃣ 모델 계수 추출
coeff = clf.coef_[0]  # 각 단어의 가중치(계수)
features = np.array(tfidf.get_feature_names_out())

# 2️⃣ 상위 긍정/부정 단어 추출
# np.argsort(): 정렬했을 때의 인덱스 반환
top_pos_idx = np.argsort(coeff)[-20:]  # 가장 큰 20개 (긍정)
top_neg_idx = np.argsort(coeff)[:20]    # 가장 작은 20개 (부정)

print("=" * 60)
print("🔍 모델이 학습한 중요 단어")
print("=" * 60)

print("\n😊 긍정적 단어 Top 20 (긍정 예측에 기여):")
pos_words = features[top_pos_idx][::-1]
pos_coeffs = coeff[top_pos_idx][::-1]
for i, (word, c) in enumerate(zip(pos_words[:10], pos_coeffs[:10]), 1):
    print(f"   {i:2d}. {word:10s} (계수: {c:.3f})")

print("\n😢 부정적 단어 Top 20 (부정 예측에 기여):")
neg_words = features[top_neg_idx]
neg_coeffs = coeff[top_neg_idx]
for i, (word, c) in enumerate(zip(neg_words[:10], neg_coeffs[:10]), 1):
    print(f"   {i:2d}. {word:10s} (계수: {c:.3f})")

# 3️⃣ 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 긍정 단어
axes[0].barh(range(15), pos_coeffs[:15], color='lightblue')
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(pos_words[:15])
axes[0].set_xlabel('계수 (Coefficient)')
axes[0].set_title('😊 긍정 예측에 기여하는 단어 Top 15')
axes[0].invert_yaxis()

# 부정 단어
axes[1].barh(range(15), neg_coeffs[:15], color='lightcoral')
axes[1].set_yticks(range(15))
axes[1].set_yticklabels(neg_words[:15])
axes[1].set_xlabel('계수 (Coefficient)')
axes[1].set_title('😢 부정 예측에 기여하는 단어 Top 15')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 🔬 7-1단계: 여러 모델 비교하기

**🎯 왜 여러 모델을 비교할까요?**
- 데이터마다 잘 맞는 모델이 다릅니다
- 베이스라인보다 좋은 모델을 찾기 위해

**📊 비교할 모델들:**
- **Naive Bayes**: 텍스트 분류에 적합한 확률 모델
- **SVM**: 고차원 데이터에 강한 모델
- **Random Forest**: 여러 결정 트리의 앙상블
- **XGBoost**: 고성능 그래디언트 부스팅

In [ ]:
# ============================================================
# 🔬 여러 머신러닝 모델 비교
# ============================================================
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import time

# 비교할 모델들 정의
models = {
    '1. Logistic Regression (베이스라인)': LogisticRegression(max_iter=3000),
    '2. Naive Bayes': MultinomialNB(),
    '3. SVM (Linear)': LinearSVC(max_iter=3000),
    '4. Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    '5. XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# 결과 저장
results = []

print("=" * 60)
print("🔬 여러 모델 성능 비교")
print("=" * 60)

for name, model in models.items():
    print(f"\n⏳ {name} 학습 중...")
    start_time = time.time()
    
    # 모델 학습
    model.fit(X_tr, y_tr)
    
    # 예측
    pred = model.predict(X_val)
    
    # 성능 계산
    acc = accuracy_score(y_val, pred)
    f1 = f1_score(y_val, pred)
    train_time = time.time() - start_time
    
    results.append({
        '모델': name,
        'Accuracy': acc,
        'F1 Score': f1,
        '학습 시간(초)': train_time
    })
    
    print(f"   ✅ 완료! Accuracy: {acc:.4f}, F1: {f1:.4f}, 시간: {train_time:.2f}초")

# 결과 정리
results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("📊 모델 성능 비교 결과")
print("=" * 60)
print(results_df.to_string(index=False))

# 최고 성능 모델 확인
best_model = results_df.loc[results_df['Accuracy'].idxmax(), '모델']
best_acc = results_df['Accuracy'].max()
print(f"\n🏆 최고 성능 모델: {best_model}")
print(f"   Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

In [ ]:
# ============================================================
# 📊 모델 성능 비교 시각화
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 모델 이름 간소화
short_names = ['LR', 'NB', 'SVM', 'RF', 'XGB']

# Accuracy 비교
colors = ['skyblue' if acc < best_acc else 'gold' for acc in results_df['Accuracy']]
axes[0].bar(short_names, results_df['Accuracy'], color=colors, edgecolor='black')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('📊 모델별 정확도 비교', fontsize=14)
axes[0].set_ylim(0.7, 0.9)
axes[0].axhline(y=results_df['Accuracy'].iloc[0], color='red', linestyle='--', 
                label=f'베이스라인: {results_df["Accuracy"].iloc[0]:.3f}')
axes[0].legend()

# 수치 표시
for i, v in enumerate(results_df['Accuracy']):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)

# 학습 시간 비교
axes[1].bar(short_names, results_df['학습 시간(초)'], color='lightgreen', edgecolor='black')
axes[1].set_ylabel('학습 시간 (초)')
axes[1].set_title('⏱️ 모델별 학습 시간 비교', fontsize=14)

# 수치 표시
for i, v in enumerate(results_df['학습 시간(초)']):
    axes[1].text(i, v + 0.1, f'{v:.1f}s', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 인사이트:")
print("   - 베이스라인(LR) 대비 성능 향상이 있는 모델을 확인하세요")
print("   - 성능과 학습 시간의 트레이드오프를 고려하세요")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 3 모델 하나 더 비교 (결정 트리) <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>DecisionTreeClassifier</code>를 <code>X_tr, y_tr</code>로 학습해 검증셋 <b>Accuracy</b>를 구하고, 위 베이스라인(로지스틱 회귀)과 견줘 보세요.<br>
<b>힌트</b> &nbsp; <code>모델.fit(X_tr, y_tr)</code> 학습 후 <code>accuracy_score(y_val, 모델.predict(X_val))</code> · 트리는 <code>random_state=42</code><br>
<b>예상</b> &nbsp; 단일 트리라 베이스라인보다 정확도가 다소 낮게 나온다
</div>
</div>

In [ ]:
# ✏️ 연습문제 3 — DecisionTreeClassifier 를 학습하고 Accuracy 를 베이스라인과 비교


## 🚀 7-2단계: 성능 개선 방향

**📈 머신러닝 성능을 높이는 방법들:**

### 1️⃣ 하이퍼파라미터 튜닝
- 모델의 설정값(하이퍼파라미터)을 조절하여 성능 향상
- 예: 로지스틱 회귀의 `C` 값, Random Forest의 `n_estimators` 등

### 2️⃣ 데이터 증강
- 더 많은 데이터 사용 (현재 10% 샘플링 → 전체 데이터 사용)
- 데이터 불균형 해결 (오버샘플링, 언더샘플링)

### 3️⃣ 특성 공학 (Feature Engineering)
- n-gram 사용 (연속된 단어 조합)
- 품사 태그 활용
- 문장 길이, 특수문자 수 등 추가 특성

### 4️⃣ 앙상블 기법
- 여러 모델의 예측을 결합
- Voting, Stacking 등

In [ ]:
# ============================================================
# 🔧 성능 개선 1: 하이퍼파라미터 튜닝
# ============================================================
# 💡 GridSearchCV: 여러 하이퍼파라미터 조합을 자동으로 테스트
from sklearn.model_selection import GridSearchCV

print("=" * 60)
print("🔧 로지스틱 회귀 하이퍼파라미터 튜닝")
print("=" * 60)

# 탐색할 하이퍼파라미터 정의
param_grid = {
    'C': [0.1, 1, 10],           # 규제 강도 (작을수록 강한 규제)
    'penalty': ['l1', 'l2'],     # 규제 종류
    'solver': ['liblinear']      # 최적화 알고리즘
}

print("📋 탐색할 파라미터:")
for k, v in param_grid.items():
    print(f"   - {k}: {v}")

# GridSearchCV 실행
print("\n⏳ 하이퍼파라미터 탐색 중... (시간이 걸릴 수 있습니다)")
grid_search = GridSearchCV(
    LogisticRegression(max_iter=3000),
    param_grid,
    cv=3,              # 3-fold 교차 검증
    scoring='accuracy',
    n_jobs=-1          # 모든 CPU 코어 사용
)
grid_search.fit(X_tr, y_tr)

# 최적 파라미터 출력
print("\n✅ 튜닝 완료!")
print(f"🏆 최적 파라미터: {grid_search.best_params_}")
print(f"📊 교차검증 최고 점수: {grid_search.best_score_:.4f}")

# 튜닝된 모델로 예측
tuned_pred = grid_search.predict(X_val)
tuned_acc = accuracy_score(y_val, tuned_pred)
tuned_f1 = f1_score(y_val, tuned_pred)

# 기존 베이스라인과 비교
baseline_acc = results_df['Accuracy'].iloc[0]
improvement = (tuned_acc - baseline_acc) * 100

print(f"\n📊 튜닝 전후 비교:")
print(f"   - 튜닝 전 (베이스라인): {baseline_acc:.4f}")
print(f"   - 튜닝 후: {tuned_acc:.4f}")
print(f"   - 성능 변화: {improvement:+.2f}%p")

In [ ]:
# ============================================================
# 🔧 성능 개선 2: n-gram 사용
# ============================================================
# 💡 n-gram이란? 연속된 n개의 단어 조합
#    - unigram (1-gram): "영화", "재밌다"
#    - bigram (2-gram): "영화 재밌다", "정말 좋다"
#    - trigram (3-gram): "영화 정말 재밌다"

print("=" * 60)
print("🔧 n-gram을 활용한 성능 개선")
print("=" * 60)

# n-gram TF-IDF 벡터화 (1-gram + 2-gram)
tfidf_ngram = TfidfVectorizer(
    tokenizer=lambda x: x,
    lowercase=False,
    max_features=30000,
    ngram_range=(1, 2)  # 1-gram과 2-gram 모두 사용
)

X_ngram = tfidf_ngram.fit_transform(sample_df['tokens'])
X_tr_ngram, X_val_ngram, y_tr_ngram, y_val_ngram = train_test_split(
    X_ngram, y, test_size=0.2, random_state=42
)

print(f"📌 기존 특성 수: {X.shape[1]:,}개")
print(f"📌 n-gram 적용 후 특성 수: {X_ngram.shape[1]:,}개")

# n-gram 적용 모델 학습
clf_ngram = LogisticRegression(max_iter=3000)
clf_ngram.fit(X_tr_ngram, y_tr_ngram)
pred_ngram = clf_ngram.predict(X_val_ngram)

ngram_acc = accuracy_score(y_val_ngram, pred_ngram)
ngram_f1 = f1_score(y_val_ngram, pred_ngram)

# 비교
print(f"\n📊 n-gram 적용 전후 비교:")
print(f"   - 기존 (1-gram만): {baseline_acc:.4f}")
print(f"   - n-gram 적용 후: {ngram_acc:.4f}")
print(f"   - 성능 변화: {(ngram_acc - baseline_acc)*100:+.2f}%p")

# n-gram 예시 출력
ngram_features = tfidf_ngram.get_feature_names_out()
bigrams = [f for f in ngram_features if ' ' in f][:10]
print(f"\n📝 학습된 bigram 예시: {bigrams}")

In [ ]:
# ============================================================
# 🔧 성능 개선 3: 앙상블 (Voting)
# ============================================================
# 💡 앙상블(Ensemble)이란? 여러 모델의 예측을 결합하는 기법
#    - "세 사람의 지혜는 한 사람보다 낫다"와 같은 원리
#    - Voting: 여러 모델의 다수결 투표

from sklearn.ensemble import VotingClassifier

print("=" * 60)
print("🔧 앙상블 (Voting Classifier)")
print("=" * 60)

# 앙상블에 사용할 모델들 정의
ensemble_models = [
    ('lr', LogisticRegression(max_iter=3000)),
    ('nb', MultinomialNB()),
    ('svm', LinearSVC(max_iter=3000))
]

# Voting Classifier 생성
# voting='hard': 다수결 투표 (가장 많이 예측한 클래스 선택)
voting_clf = VotingClassifier(estimators=ensemble_models, voting='hard')

print("📋 앙상블에 포함된 모델:")
for name, _ in ensemble_models:
    print(f"   - {name}")

# 학습 및 예측
print("\n⏳ 앙상블 모델 학습 중...")
voting_clf.fit(X_tr, y_tr)
voting_pred = voting_clf.predict(X_val)

voting_acc = accuracy_score(y_val, voting_pred)
voting_f1 = f1_score(y_val, voting_pred)

print("✅ 학습 완료!")
print(f"\n📊 앙상블 결과:")
print(f"   - 앙상블 Accuracy: {voting_acc:.4f}")
print(f"   - 앙상블 F1 Score: {voting_f1:.4f}")
print(f"\n📊 베이스라인 대비:")
print(f"   - 베이스라인: {baseline_acc:.4f}")
print(f"   - 앙상블: {voting_acc:.4f}")
print(f"   - 성능 변화: {(voting_acc - baseline_acc)*100:+.2f}%p")

## 🔬 7-1단계: 여러 모델 비교하기

**🎯 왜 여러 모델을 비교할까요?**
- 데이터마다 잘 맞는 모델이 다릅니다
- 베이스라인보다 좋은 모델을 찾기 위해
- 각 모델의 장단점을 이해하기 위해

**📊 비교할 모델들:**
| 모델 | 특징 | 장점 | 단점 |
|------|------|------|------|
| **Logistic Regression** | 선형 분류 | 빠름, 해석 쉬움 | 복잡한 패턴 학습 어려움 |
| **Naive Bayes** | 확률 기반 | 매우 빠름, 텍스트에 적합 | 독립 가정이 비현실적 |
| **SVM** | 마진 최대화 | 고차원에 강함 | 느림 |
| **Random Forest** | 앙상블 | 과적합에 강함 | 해석 어려움 |
| **XGBoost** | 그래디언트 부스팅 | 높은 성능 | 튜닝 필요 |

## 🎯 7단계: 실제 예측 해보기

**🔮 예측 함수 만들기**   
위에서 만든 함수를 사용해서 새로운 리뷰의 감성을 예측해보기

**✨ 테스트해보기:**
- "이 영화는 정말 재밌네요!" → 긍정 (99.0% 확률)
- 다른 문장도 직접 테스트해보세요!

In [ ]:
# ============================================================
# 🎯 새로운 리뷰 예측하기
# ============================================================
# 💡 학습된 모델을 사용해서 새로운 텍스트의 감성을 예측합니다

def predict_sentiment(text, verbose=True):
    """
    텍스트의 감성을 예측하는 함수
    
    Parameters:
        text: 분석할 텍스트
        verbose: 상세 출력 여부
    Returns:
        result: '긍정' 또는 '부정'
        confidence: 예측 확률
    """
    # 1. 토큰화
    tokens = tokenize(text)
    
    # 2. TF-IDF 벡터화
    X_new = tfidf.transform([tokens])
    
    # 3. 예측
    pred = clf.predict(X_new)[0]
    prob = clf.predict_proba(X_new)[0]
    
    # 4. 결과 정리
    if pred == 1:
        result = '긍정 😊'
        confidence = prob[1]
    else:
        result = '부정 😢'
        confidence = prob[0]
    
    if verbose:
        print(f"📝 입력: \"{text}\"")
        print(f"🔤 토큰: {tokens}")
        print(f"🎯 결과: {result} (확신도: {confidence:.1%})")
        print("-" * 50)
    
    return result, confidence

# 여러 예시 테스트
print("=" * 60)
print("🎯 감성 분석 테스트")
print("=" * 60)
print()

test_texts = [
    "이 영화는 정말 재밌네요! 강력 추천합니다.",
    "시간 아깝다. 최악의 영화.",
    "그냥 그래요. 볼만은 합니다.",
    "배우들의 연기가 훌륭했어요.",
    "스토리가 너무 뻔하고 지루했다.",
]

for text in test_texts:
    predict_sentiment(text)
    print()

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 4 내 문장으로 감성 예측 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>predict_sentiment()</code> 함수로 <b>내가 직접 쓴 리뷰 2개</b>(긍정 1개·부정 1개)의 감성을 예측해 보세요.<br>
<b>힌트</b> &nbsp; 함수 호출만으로 결과·확신도가 출력됨 <code>predict_sentiment('문장')</code><br>
<b>예상</b> &nbsp; 긍정 문장은 '긍정 😊', 부정 문장은 '부정 😢'으로 분류
</div>
</div>

In [ ]:
# ✏️ 연습문제 4 — 내가 쓴 리뷰 2개의 감성을 predict_sentiment 로 예측


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🎯 종합문제  긍정·부정 핵심 단어 뽑기 <span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">종합</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 베이스라인 모델 계수(<code>clf.coef_[0]</code>)에서 <b>가장 긍정적인 단어</b>와 <b>가장 부정적인 단어</b>를 하나씩 출력하세요.<br>
<b>힌트</b> &nbsp; 특성 이름 <code>features = np.array(tfidf.get_feature_names_out())</code> · 최대/최소 위치 <code>np.argmax</code>·<code>np.argmin</code><br>
<b>예상</b> &nbsp; 긍정엔 칭찬 단어, 부정엔 혹평 단어가 나온다
</div>
</div>

In [ ]:
# 🎯 종합문제 — 처음부터 직접 작성해 보세요


---

## 📝 전체 내용 정리

### 🎯 감성 분석 프로세스

| 단계 | 과정 | 핵심 기술 | 사용한 코드/라이브러리 |
|------|------|-----------|----------------------|
| **1단계** | 환경 설정 | 라이브러리 설치 | `pip install` |
| **2단계** | 데이터 수집 | NSMC 영화 리뷰 | `pd.read_csv()` |
| **3단계** | 데이터 탐색 | 통계 분석, 시각화 | `df.describe()`, `plt` |
| **4단계** | 텍스트 전처리 | 형태소 분석, 불용어 제거 | `konlpy.Okt()` |
| **5단계** | 시각화 | 워드클라우드 | `WordCloud` |
| **6단계** | 벡터화 | TF-IDF 변환 | `TfidfVectorizer` |
| **7단계** | 모델 훈련 | 로지스틱 회귀 (베이스라인) | `LogisticRegression` |
| **7-1단계** | 모델 비교 | 여러 알고리즘 비교 | `NB`, `SVM`, `RF`, `XGB` |
| **7-2단계** | 성능 개선 | 튜닝, n-gram, 앙상블 | `GridSearchCV`, `VotingClassifier` |
| **8단계** | 예측 | 새로운 텍스트 분류 | `predict_sentiment()` |

### 💡 핵심 개념들

✅ **자연어 처리 (NLP)**: 컴퓨터가 인간의 언어를 이해하는 기술  
✅ **형태소 분석**: 문장을 의미있는 단위로 분해  
✅ **TF-IDF**: 단어의 중요도를 수치화하는 방법  
✅ **머신러닝**: 데이터로부터 패턴을 학습하는 기술  
✅ **베이스라인**: 성능 비교의 기준이 되는 기본 모델  
✅ **하이퍼파라미터 튜닝**: 모델 설정값 최적화  
✅ **앙상블**: 여러 모델의 예측을 결합하는 기법  

### 📈 성능 개선 방법 요약

| 방법 | 설명 | 효과 |
|------|------|------|
| **하이퍼파라미터 튜닝** | 모델 설정값 최적화 | 약간의 성능 향상 |
| **n-gram** | 연속 단어 조합 사용 | 문맥 정보 포착 |
| **앙상블** | 여러 모델 결합 | 안정적인 예측 |
| **더 많은 데이터** | 전체 데이터 사용 | 일반화 성능 향상 |

### 🚀 실무 활용 방안

1. **고객 리뷰 분석**: 제품/서비스 만족도 자동 분류
2. **소셜미디어 모니터링**: 브랜드 언급 감성 분석
3. **고객 지원**: 문의 내용의 긴급도/감성 자동 판단
4. **시장 조사**: 신제품에 대한 여론 분석

### 🔜 더 공부하기

- **딥러닝 모델**: BERT, GPT 등 트랜스포머 모델
- **Word2Vec/FastText**: 단어 임베딩 기법
- **감성 사전**: KNU 한국어 감성사전 활용

---

## 🎉 수고하셨습니다!
오늘 머신러닝으로 영화 리뷰 감성 분석을 완료했습니다.  
배운 내용을 응용해서 다양한 텍스트 분류 문제에 도전해보세요!